[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/095.%20The%20Supply%20Chain%20Network%20Design%20under%20Uncertainty/P95-Tier-2_executed.ipynb)

---

# 95. The Supply Chain Network Design under Uncertainty Problem
## Tier 2: Scenario-Weighted Cheapest Insertion Heuristic

### Purpose and scope

This notebook builds a facility-location solution one customer at a time. At each step, the heuristic chooses the feasible customer–facility pair with the smallest incremental score: the customer's expected transportation cost plus the facility's fixed cost if that facility has not yet been opened. Demand uncertainty is represented by a finite set of scenarios and their probabilities.

The inputs are facility fixed costs and capacities, customer demand in each scenario, facility–customer unit transportation costs, and scenario probabilities. The outputs are the opened facilities, customer assignments, scenario-specific shipments, expected cost, and construction record. A small mixed-integer model is also used to compare the heuristic with an exact solution on a smaller instance under the same assignment policy.

### Model assumptions

- Each customer is assigned in full to exactly one facility, and that facility serves the customer in every scenario. Shipments equal the customer's demand in the realized scenario; customers are not split across facilities.
- Facility capacity is a per-scenario limit. Loads in different scenarios are not added together because the scenarios represent alternative demand conditions, not simultaneous operating periods.
- Facility fixed costs are paid once if a facility is opened. Transportation costs are linear in shipped quantity, and the objective minimizes expected total cost.
- The heuristic makes irrevocable assignments. It does not backtrack or reassign customers, so it can fail to complete an instance even when a feasible complete assignment exists.
- The model excludes such details as facility-opening lead times, service-level requirements, multiple products, and uncertainty in costs or capacities.

### Mathematical model and heuristic decision rule

Let $I$ be the set of facilities, $J$ the set of customers, and $S$ the set of demand scenarios. Let $f_i$ be the fixed cost of facility $i$, $K_i$ its capacity, $p_s$ the probability of scenario $s$, $d_{js}$ the demand of customer $j$ in scenario $s$, and $c_{ij}$ the unit transportation cost from facility $i$ to customer $j$.

The common-assignment policy is represented by binary variable $z_{ij}$, which is 1 when customer $j$ is assigned to facility $i$. Binary variable $y_i$ indicates whether facility $i$ is open, and $x_{ijs}$ is the quantity shipped from facility $i$ to customer $j$ in scenario $s$:

$$
\begin{aligned}
\min\quad & \sum_{i\in I} f_i y_i + \sum_{s\in S} p_s \sum_{i\in I}\sum_{j\in J} c_{ij}x_{ijs} \\
\text{subject to}\quad
& \sum_{i\in I} z_{ij}=1 && j\in J, \\
& \sum_{i\in I} x_{ijs}=d_{js} && j\in J,\ s\in S, \\
& \sum_{j\in J} x_{ijs}\le K_i y_i && i\in I,\ s\in S, \\
& x_{ijs}\le d_{js} z_{ij} && i\in I,\ j\in J,\ s\in S, \\
& z_{ij}\le y_i && i\in I,\ j\in J, \\
& y_i\in\{0,1\},\quad z_{ij}\in\{0,1\},\quad x_{ijs}\ge 0.
\end{aligned}
$$

Here, $y_i$ is the facility-opening decision, $z_{ij}$ is the customer-to-facility assignment decision, and $x_{ijs}$ is the scenario-specific shipment quantity. The first objective term charges each opened facility once; the second is expected transportation cost. The assignment constraint gives every customer exactly one facility. Demand is met separately in every scenario, and each facility's capacity is checked separately in every scenario. The bound $x_{ijs}\le d_{js}z_{ij}$ links shipments to the common customer assignment. Since shipments are nonnegative and must sum to $d_{js}$ for each customer and scenario, $d_{js}$ is a valid scenario-specific upper bound. The assignment variables have no scenario index: this is what makes a customer's facility the same across all scenarios. In the code, `open_facility`, `assign`, and `shipment` represent $y_i$, $z_{ij}$, and $x_{ijs}$, respectively.

The heuristic constructs a feasible partial solution. At iteration $t$, let $U_t$ be the set of unassigned customers, $O_t$ the set of opened facilities, and $L_{is}^{(t)}$ the current load of facility $i$ in scenario $s$. A candidate pair $(i,j)$ is feasible when

$$
L_{is}^{(t)}+d_{js}\le K_i\qquad\text{for every }s\in S.
$$

Among feasible pairs, it selects the smallest score

$$
g(i,j)=\sum_{s\in S}p_s c_{ij}d_{js} + \begin{cases} f_i, & i\notin O_t,\\ 0, & i\in O_t.\end{cases}
$$

The first term is the customer's expected transportation cost at facility $i$; the second charges the fixed cost only when opening a new facility. Formally, if $F_t$ is the set of feasible pairs at iteration $t$, the selected pair is

$$
(i_t,j_t)\in\underset{(i,j)\in F_t}{\operatorname{argmin}}\ g(i,j).
$$

If $F_t$ is empty, construction stops with a partial solution. Otherwise, the update is $U_{t+1}=U_t\setminus\{j_t\}$, $O_{t+1}=O_t\cup\{i_t\}$, and $L_{i_t s}^{(t+1)}=L_{i_t s}^{(t)}+d_{j_t s}$ for every scenario $s$; all other facility loads remain unchanged. The base state has $U_0=J$, no opened facilities, and zero loads. This is an iterative greedy construction, not a recursive algorithm: each iteration commits the lowest-scoring feasible pair and does not revisit earlier choices. The rule has no optimality guarantee because a locally attractive assignment can make later assignments impossible.

With $|J|$ customers, $|I|$ facilities, and $|S|$ scenarios, a direct implementation evaluates up to $|J||I|$ pairs at each of at most $|J|$ iterations. Each evaluation may inspect all scenarios, giving a worst-case time complexity of $O(|J|^2|I||S|)$ and a space requirement dominated by the input and assignment records. The code exposes the cost calculation, capacity check, candidate selection, assignment update, and total-cost calculation as separate methods so each part of the rule can be inspected.

### What to examine

- Whether the construction completes, and which facility–customer pairs are selected.
- How fixed and expected transportation costs contribute to the total.
- Whether every opened facility respects capacity in every scenario.
- How the heuristic compares with a capacity-relaxed reference and, on a smaller instance, with an exact model using the same assignment policy.

A capacity-relaxed reference assigns each customer to its lowest expected-transport-cost facility and then charges the fixed costs of the facilities it uses. It deliberately ignores capacity, so it is a reference value rather than a feasible solution or an optimality benchmark. The exact comparison is limited to its specified smaller instance; its results do not establish a general solution-quality or runtime guarantee.

In [ ]:
from dataclasses import dataclass
from itertools import product
import math
import time
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the instance

The instance stores the data needed by the objective, the scenario-by-scenario capacity checks, and the construction procedure. Scenario indices identify alternative demand conditions; their probabilities must sum to one so that the weighted transportation cost is an expected cost. Facility and customer indices in this example are zero-based and are used to index the fixed-cost and capacity lists. The demand dictionary includes a product index; the example uses a single product, represented by index 0.

In [ ]:
@dataclass
class HeuristicInstance:
    """Data for a scenario-weighted supply chain network design instance."""

    facilities: List[int]
    customers: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    demands: Dict[Tuple[int, int, int], float]
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int], float]
    scenario_probs: List[float]


def create_heuristic_instance() -> HeuristicInstance:
    """Create the five-facility, eight-customer teaching instance."""
    facilities = [0, 1, 2, 3, 4]
    customers = [0, 1, 2, 3, 4, 5, 6, 7]
    scenarios = [0, 1, 2]

    fixed_costs = [150, 180, 120, 200, 160]
    capacities = [120, 150, 100, 180, 140]
    scenario_probs = [0.3, 0.5, 0.2]

    transport_costs = {
        (0, 0): 2.5, (0, 1): 3.0, (0, 2): 3.5, (0, 3): 4.0,
        (0, 4): 3.8, (0, 5): 2.8, (0, 6): 4.2, (0, 7): 3.3,
        (1, 0): 3.2, (1, 1): 2.1, (1, 2): 4.1, (1, 3): 3.6,
        (1, 4): 2.9, (1, 5): 3.4, (1, 6): 3.7, (1, 7): 2.6,
        (2, 0): 4.1, (2, 1): 4.6, (2, 2): 2.0, (2, 3): 2.4,
        (2, 4): 4.2, (2, 5): 3.9, (2, 6): 2.8, (2, 7): 3.1,
        (3, 0): 2.7, (3, 1): 2.4, (3, 2): 2.9, (3, 3): 2.1,
        (3, 4): 3.1, (3, 5): 2.6, (3, 6): 3.3, (3, 7): 2.2,
        (4, 0): 3.5, (4, 1): 3.8, (4, 2): 3.2, (4, 3): 2.8,
        (4, 4): 2.3, (4, 5): 3.7, (4, 6): 2.9, (4, 7): 2.5,
    }

    high_demand = [80, 70, 60, 90, 75, 85, 65, 95]
    medium_demand = [60, 50, 45, 70, 55, 65, 50, 75]
    low_demand = [40, 35, 30, 50, 40, 45, 35, 55]
    demand_by_scenario = [high_demand, medium_demand, low_demand]

    demands = {
        (customer, 0, scenario): demand_by_scenario[scenario][customer]
        for customer in customers
        for scenario in scenarios
    }

    return HeuristicInstance(
        facilities=facilities,
        customers=customers,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        demands=demands,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
    )


heuristic_instance = create_heuristic_instance()

assert math.isclose(sum(heuristic_instance.scenario_probs), 1.0)
assert len(heuristic_instance.fixed_costs) == len(heuristic_instance.facilities)
assert len(heuristic_instance.capacities) == len(heuristic_instance.facilities)

print("Heuristic instance:")
print(f"Facilities: {len(heuristic_instance.facilities)}")
print(f"Customers: {len(heuristic_instance.customers)}")
print(f"Scenarios: {len(heuristic_instance.scenarios)}")
print(f"Fixed costs: {heuristic_instance.fixed_costs}")
print(f"Capacities: {heuristic_instance.capacities}")
print(f"Scenario probabilities: {heuristic_instance.scenario_probs}")

## 2. Construct a solution with cheapest insertion

The construction keeps one load value for each facility–scenario pair. This representation is essential: a facility must have enough capacity in each possible scenario, but demand in mutually exclusive scenarios is not added together. For a candidate customer, the capacity check examines every scenario before the heuristic considers its cost.

At each iteration, `find_best_assignment` scans the unassigned customers and facilities in their listed order. Strictly lower scores replace the current best candidate, so ties are resolved by that deterministic scan order. `make_assignment` commits the selected customer to the same facility in every scenario. The process stops when all customers are assigned or when no remaining pair is feasible.

In [ ]:
class CheapestInsertionHeuristic:
    """Construct a scenario-weighted facility assignment greedily."""

    def __init__(self, instance: HeuristicInstance):
        self.instance = instance
        self.opened_facilities: Set[int] = set()
        self.assignments: Dict[Tuple[int, int], int] = {}
        self.shipments: Dict[Tuple[int, int, int], float] = {}
        self.facility_load: Dict[Tuple[int, int], float] = {}

    def calculate_scenario_weighted_cost(
        self, facility: int, customer: int
    ) -> float:
        """Return the customer's expected transport cost at a facility."""
        return sum(
            self.instance.scenario_probs[scenario]
            * self.instance.transport_costs[(facility, customer)]
            * self.instance.demands[(customer, 0, scenario)]
            for scenario in self.instance.scenarios
        )

    def check_capacity_feasibility(
        self, facility: int, customer: int
    ) -> bool:
        """Check the candidate addition against capacity in every scenario."""
        for scenario in self.instance.scenarios:
            current_load = self.facility_load.get((facility, scenario), 0.0)
            demand = self.instance.demands[(customer, 0, scenario)]
            if current_load + demand > self.instance.capacities[facility]:
                return False
        return True

    def find_best_assignment(
        self, unassigned_customers: List[int]
    ) -> Tuple[Optional[int], Optional[int], float]:
        """Find the feasible customer–facility pair with the lowest score."""
        best_cost = float("inf")
        best_facility = None
        best_customer = None

        for customer in unassigned_customers:
            for facility in self.instance.facilities:
                if not self.check_capacity_feasibility(facility, customer):
                    continue

                cost = self.calculate_scenario_weighted_cost(facility, customer)
                if facility not in self.opened_facilities:
                    cost += self.instance.fixed_costs[facility]

                if cost < best_cost:
                    best_cost = cost
                    best_facility = facility
                    best_customer = customer

        return best_facility, best_customer, best_cost

    def make_assignment(self, facility: int, customer: int) -> None:
        """Commit the customer to one facility across all scenarios."""
        self.opened_facilities.add(facility)

        for scenario in self.instance.scenarios:
            demand = self.instance.demands[(customer, 0, scenario)]
            self.assignments[(customer, scenario)] = facility
            self.shipments[(facility, customer, scenario)] = demand
            load_key = (facility, scenario)
            self.facility_load[load_key] = self.facility_load.get(load_key, 0.0) + demand

    def solve(self) -> Dict:
        """Run the greedy construction and return its solution record."""
        # Reset state so repeated calls solve the same instance independently.
        self.opened_facilities.clear()
        self.assignments.clear()
        self.shipments.clear()
        self.facility_load.clear()

        start_time = time.perf_counter()
        unassigned_customers = self.instance.customers.copy()
        construction_steps = []
        iteration = 0

        while unassigned_customers:
            iteration += 1
            facility, customer, cost = self.find_best_assignment(
                unassigned_customers
            )

            if facility is None or customer is None:
                print(
                    "No feasible customer–facility assignment is available "
                    f"at iteration {iteration}."
                )
                break

            is_new_facility = facility not in self.opened_facilities
            self.make_assignment(facility, customer)
            unassigned_customers.remove(customer)

            construction_steps.append(
                {
                    "iteration": iteration,
                    "customer": customer,
                    "facility": facility,
                    "cost": cost,
                    "opened_facilities": len(self.opened_facilities),
                    "newly_opened_facility": facility if is_new_facility else None,
                    "remaining_customers": len(unassigned_customers),
                }
            )

        solution_time = time.perf_counter() - start_time
        return {
            "opened_facilities": sorted(self.opened_facilities),
            "assignments": self.assignments.copy(),
            "shipments": self.shipments.copy(),
            "total_cost": self.calculate_total_cost(),
            "construction_steps": construction_steps,
            "solution_time": solution_time,
            "iterations": iteration,
            "unassigned_customers": unassigned_customers.copy(),
            "complete": not unassigned_customers,
        }

    def calculate_total_cost(self) -> float:
        """Calculate fixed cost plus probability-weighted transport cost."""
        fixed_cost = sum(
            self.instance.fixed_costs[facility]
            for facility in self.opened_facilities
        )
        variable_cost = sum(
            self.instance.scenario_probs[scenario]
            * self.instance.transport_costs[(facility, customer)]
            * quantity
            for (facility, customer, scenario), quantity in self.shipments.items()
        )
        return fixed_cost + variable_cost


heuristic = CheapestInsertionHeuristic(heuristic_instance)
heuristic_solution = heuristic.solve()

print("\nHeuristic solution results:")
print(f"Opened facilities: {heuristic_solution['opened_facilities']}")
print(f"Assigned customers: {len(heuristic_solution['assignments']) // len(heuristic_instance.scenarios)}")
print(f"Unassigned customers: {heuristic_solution['unassigned_customers']}")
print(f"Complete solution: {heuristic_solution['complete']}")
print(f"Expected cost of recorded assignments: ${heuristic_solution['total_cost']:,.2f}")
print(f"Construction time: {heuristic_solution['solution_time']:.6f} seconds")
print(f"Construction iterations: {heuristic_solution['iterations']}")

## 3. Inspect and validate the constructed solution

The cost reported for a complete solution consists of fixed costs for opened facilities and expected transportation costs for the assigned demand. The scenario-specific utilization report checks the capacity interpretation directly: each facility's shipped quantity is compared with its capacity separately in each scenario. A partial construction is reported as partial; its cost is not presented as the cost of a complete feasible network.

In [ ]:
def analyze_heuristic_solution(
    solution: Dict, instance: HeuristicInstance
) -> None:
    """Report cost components, assignments, loads, and construction steps."""
    print("\nDetailed solution analysis:")

    fixed_cost = sum(
        instance.fixed_costs[facility]
        for facility in solution["opened_facilities"]
    )
    variable_cost = solution["total_cost"] - fixed_cost
    total_cost = solution["total_cost"]

    print("\nCost breakdown:")
    if math.isclose(total_cost, 0.0, abs_tol=1e-12):
        print(f"  Fixed cost: ${fixed_cost:,.2f}")
        print(f"  Expected transportation cost: ${variable_cost:,.2f}")
        print(f"  Total recorded cost: ${total_cost:,.2f}")
    else:
        print(
            f"  Fixed cost: ${fixed_cost:,.2f} "
            f"({fixed_cost / total_cost * 100:.1f}%)"
        )
        print(
            f"  Expected transportation cost: ${variable_cost:,.2f} "
            f"({variable_cost / total_cost * 100:.1f}%)"
        )
        print(f"  Total recorded cost: ${total_cost:,.2f}")

    scenario_names = {
        0: "High demand",
        1: "Medium demand",
        2: "Low demand",
    }
    print("\nAssignments by scenario:")
    for scenario in instance.scenarios:
        name = scenario_names.get(scenario, f"Scenario {scenario}")
        print(
            f"\n{name} (probability {instance.scenario_probs[scenario]:.2f}):"
        )
        for facility in solution["opened_facilities"]:
            assigned_customers = [
                customer
                for customer in instance.customers
                if solution["assignments"].get((customer, scenario)) == facility
            ]
            if not assigned_customers:
                continue
            quantity = sum(
                solution["shipments"][(facility, customer, scenario)]
                for customer in assigned_customers
            )
            transport_cost = sum(
                instance.transport_costs[(facility, customer)]
                * solution["shipments"][(facility, customer, scenario)]
                for customer in assigned_customers
            )
            print(
                f"  Facility {facility}: customers {assigned_customers}, "
                f"quantity {quantity:.0f}, transportation cost "
                f"${transport_cost:,.2f}"
            )

    print("\nScenario-specific capacity utilization:")
    for facility in solution["opened_facilities"]:
        capacity = instance.capacities[facility]
        for scenario in instance.scenarios:
            load = sum(
                solution["shipments"].get((facility, customer, scenario), 0.0)
                for customer in instance.customers
            )
            utilization = load / capacity * 100 if capacity else float("inf")
            print(
                f"  Facility {facility}, {scenario_names.get(scenario, f'Scenario {scenario}')}: "
                f"load {load:.0f} / capacity {capacity:.0f} "
                f"({utilization:.1f}%)"
            )
            assert load <= capacity + 1e-9

    steps = solution["construction_steps"]
    print("\nConstruction record:")
    print(f"  Assignments made: {len(steps)}")
    print(f"  Construction iterations: {solution['iterations']}")
    for step in steps:
        opening_note = (
            f", opened facility {step['newly_opened_facility']}"
            if step["newly_opened_facility"] is not None
            else ""
        )
        print(
            f"  Iteration {step['iteration']}: customer {step['customer']} "
            f"assigned to facility {step['facility']}; "
            f"selection score ${step['cost']:,.2f}{opening_note}"
        )

    if solution["complete"]:
        assert len(solution["assignments"]) == (
            len(instance.customers) * len(instance.scenarios)
        )
        assert math.isclose(
            solution["total_cost"],
            fixed_cost + variable_cost,
            rel_tol=1e-10,
            abs_tol=1e-8,
        )


analyze_heuristic_solution(heuristic_solution, heuristic_instance)

## 4. Visualize construction and cost structure

The construction plot separates each selected pair's score from the cumulative sum of those scores. This cumulative sum includes a facility's fixed cost at the step when it opens, so it matches the recorded cost of the assignments made. It is the final objective value when construction is complete. The capacity-relaxed reference is explicitly labeled because it may violate capacity; if construction is partial, its recorded cost and the reference do not represent comparable complete solutions. The utilization chart and cost breakdown describe the recorded heuristic assignments, not a separate optimization method.

In [ ]:
def visualize_heuristic_solution(
    solution: Dict, instance: HeuristicInstance
) -> None:
    """Plot construction, a capacity-relaxed reference, and solution details."""
    figure, axes = plt.subplots(2, 2, figsize=(15, 10))
    figure.suptitle(
        "Scenario-Weighted Cheapest Insertion: Solution Summary",
        fontsize=15,
        fontweight="bold",
    )

    steps = solution["construction_steps"]
    iterations = [step["iteration"] for step in steps]
    step_costs = [step["cost"] for step in steps]
    cumulative_costs = np.cumsum(step_costs)

    ax1 = axes[0, 0]
    ax1_twin = ax1.twinx()
    line_cost = ax1.plot(
        iterations,
        step_costs,
        "o-",
        color="#d95f02",
        label="Selected-pair score",
    )
    line_cumulative = ax1.plot(
        iterations,
        cumulative_costs,
        "s--",
        color="#1b9e77",
        label="Cumulative score",
    )
    open_counts = [step["opened_facilities"] for step in steps]
    line_open = ax1_twin.plot(
        iterations,
        open_counts,
        "^-",
        color="#7570b3",
        label="Open facilities",
    )
    ax1.set_xlabel("Construction iteration")
    ax1.set_ylabel("Cost score ($)")
    ax1_twin.set_ylabel("Number of open facilities")
    ax1.set_title("Greedy construction")
    ax1.grid(True, alpha=0.3)
    lines = line_cost + line_cumulative + line_open
    ax1.legend(lines, [line.get_label() for line in lines], loc="best")

    baseline_transport_cost = 0.0
    baseline_assignments = {}
    for customer in instance.customers:
        facility_costs = {
            facility: sum(
                instance.scenario_probs[scenario]
                * instance.transport_costs[(facility, customer)]
                * instance.demands[(customer, 0, scenario)]
                for scenario in instance.scenarios
            )
            for facility in instance.facilities
        }
        best_facility = min(facility_costs, key=facility_costs.get)
        baseline_assignments[customer] = best_facility
        baseline_transport_cost += facility_costs[best_facility]

    baseline_opened = set(baseline_assignments.values())
    baseline_fixed_cost = sum(
        instance.fixed_costs[facility] for facility in baseline_opened
    )
    baseline_total = baseline_fixed_cost + baseline_transport_cost

    ax2 = axes[0, 1]
    heuristic_label = (
        "Heuristic"
        if solution["complete"]
        else "Heuristic\n(partial; not comparable)"
    )
    comparison_labels = [
        heuristic_label,
        "Capacity-relaxed\nreference",
    ]
    comparison_costs = [solution["total_cost"], baseline_total]
    bars = ax2.bar(
        comparison_labels,
        comparison_costs,
        color=["#1b9e77", "#d95f02"],
        alpha=0.85,
    )
    ax2.set_ylabel("Expected cost ($)")
    ax2.set_title("Recorded cost; reference ignores capacity")
    ax2.grid(axis="y", alpha=0.3)
    label_offset = max(comparison_costs, default=0.0) * 0.01
    for bar, cost in zip(bars, comparison_costs):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            f"${cost:,.0f}",
            ha="center",
            va="bottom",
        )

    ax3 = axes[1, 0]
    scenario_names = {
        0: "High",
        1: "Medium",
        2: "Low",
    }
    active_facilities = solution["opened_facilities"]
    x_positions = np.arange(len(active_facilities))
    width = 0.8 / max(len(instance.scenarios), 1)
    for scenario_index, scenario in enumerate(instance.scenarios):
        utilizations = []
        for facility in active_facilities:
            load = sum(
                solution["shipments"].get((facility, customer, scenario), 0.0)
                for customer in instance.customers
            )
            capacity = instance.capacities[facility]
            utilizations.append(100 * load / capacity if capacity else 0.0)
        ax3.bar(
            x_positions + scenario_index * width,
            utilizations,
            width,
            label=f"{scenario_names.get(scenario, f'Scenario {scenario}')} demand",
        )
    ax3.set_xlabel("Facility")
    ax3.set_ylabel("Capacity utilization (%)")
    ax3.set_title("Scenario-specific capacity utilization")
    ax3.set_xticks(x_positions + width * (len(instance.scenarios) - 1) / 2)
    ax3.set_xticklabels([f"Facility {facility}" for facility in active_facilities])
    ax3.legend()
    ax3.grid(axis="y", alpha=0.3)

    ax4 = axes[1, 1]
    fixed_cost = sum(
        instance.fixed_costs[facility]
        for facility in solution["opened_facilities"]
    )
    expected_transport_cost = solution["total_cost"] - fixed_cost
    ax4.bar(
        ["Recorded solution"],
        [fixed_cost],
        label="Facility fixed cost",
        color="#7570b3",
    )
    ax4.bar(
        ["Recorded solution"],
        [expected_transport_cost],
        bottom=[fixed_cost],
        label="Expected transportation cost",
        color="#1b9e77",
    )
    ax4.set_ylabel("Cost ($)")
    ax4.set_title("Expected-cost components")
    ax4.legend()
    ax4.grid(axis="y", alpha=0.3)

    figure.tight_layout()
    plt.show()


visualize_heuristic_solution(heuristic_solution, heuristic_instance)

## 5. Compare with an exact model on a smaller instance

An exact model can check whether the greedy construction's decisions are optimal for a small instance. A fair comparison requires both approaches to use the same policy: each customer has one facility assignment shared across scenarios, while capacity and shipment quantities are evaluated scenario by scenario. The binary assignment variable in the exact model therefore has no scenario index.

The exact model is a mixed-integer linear program. It is included as a small-instance benchmark, not as a replacement for the heuristic's construction method. The comparison reports results only when the solver returns an optimal solution and the heuristic completes the assignments.

In [ ]:
def compare_with_exact_method() -> None:
    """Compare the heuristic with a matching exact model on a small instance."""
    print("\nComparison with an exact model:")

    facilities = [0, 1, 2]
    customers = [0, 1, 2, 3]
    scenarios = [0, 1]
    fixed_costs = [100, 120, 90]
    capacities = [80, 90, 70]
    scenario_probs = [0.5, 0.5]

    transport_costs = {
        (0, 0): 2.5, (0, 1): 3.0, (0, 2): 3.5, (0, 3): 4.0,
        (1, 0): 3.2, (1, 1): 2.1, (1, 2): 4.1, (1, 3): 3.6,
        (2, 0): 4.1, (2, 1): 4.6, (2, 2): 2.0, (2, 3): 2.4,
    }
    high_demand = [50, 40, 35, 45]
    low_demand = [30, 25, 20, 25]
    demand_by_scenario = [high_demand, low_demand]
    demands = {
        (customer, 0, scenario): demand_by_scenario[scenario][customer]
        for customer in customers
        for scenario in scenarios
    }

    small_instance = HeuristicInstance(
        facilities=facilities,
        customers=customers,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        demands=demands,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
    )

    problem = pulp.LpProblem("Exact_Supply_Chain", pulp.LpMinimize)
    open_facility = {
        facility: pulp.LpVariable(
            f"open_{facility}", cat=pulp.LpBinary
        )
        for facility in facilities
    }
    assign = {
        (facility, customer): pulp.LpVariable(
            f"assign_{facility}_{customer}", cat=pulp.LpBinary
        )
        for facility in facilities
        for customer in customers
    }
    shipment = {
        (facility, customer, scenario): pulp.LpVariable(
            f"shipment_{facility}_{customer}_{scenario}", lowBound=0
        )
        for facility in facilities
        for customer in customers
        for scenario in scenarios
    }

    fixed_cost_expression = pulp.lpSum(
        fixed_costs[facility] * open_facility[facility]
        for facility in facilities
    )
    expected_transport_expression = pulp.lpSum(
        scenario_probs[scenario]
        * transport_costs[(facility, customer)]
        * shipment[(facility, customer, scenario)]
        for facility in facilities
        for customer in customers
        for scenario in scenarios
    )
    problem += fixed_cost_expression + expected_transport_expression

    for customer in customers:
        problem += pulp.lpSum(
            assign[(facility, customer)] for facility in facilities
        ) == 1
        for scenario in scenarios:
            problem += pulp.lpSum(
                shipment[(facility, customer, scenario)]
                for facility in facilities
            ) == demands[(customer, 0, scenario)]

    for facility in facilities:
        for scenario in scenarios:
            problem += pulp.lpSum(
                shipment[(facility, customer, scenario)]
                for customer in customers
            ) <= capacities[facility] * open_facility[facility]

        for customer in customers:
            problem += assign[(facility, customer)] <= open_facility[facility]
            for scenario in scenarios:
                problem += (
                    shipment[(facility, customer, scenario)]
                    <= demands[(customer, 0, scenario)]
                    * assign[(facility, customer)]
                )

    exact_start = time.perf_counter()
    problem.solve(pulp.PULP_CBC_CMD(msg=False))
    exact_time = time.perf_counter() - exact_start
    solver_status = pulp.LpStatus[problem.status]
    print(f"Exact model status: {solver_status}")

    if solver_status != "Optimal":
        print("The solver did not certify an optimal solution; comparison is omitted.")
        return

    exact_cost = pulp.value(problem.objective)
    if exact_cost is None:
        print("The solver returned no objective value; comparison is omitted.")
        return

    heuristic_start = time.perf_counter()
    small_heuristic = CheapestInsertionHeuristic(small_instance)
    small_solution = small_heuristic.solve()
    heuristic_time = time.perf_counter() - heuristic_start

    if not small_solution["complete"]:
        print(
            "The heuristic did not complete this instance; its partial cost "
            "is not compared with the exact complete-solution objective."
        )
        return

    heuristic_cost = small_solution["total_cost"]
    gap_percent = (
        (heuristic_cost - exact_cost) / abs(exact_cost) * 100
        if not math.isclose(exact_cost, 0.0, abs_tol=1e-12)
        else float("nan")
    )

    print(f"Exact objective: ${exact_cost:,.2f}")
    print(f"Exact solve time: {exact_time:.6f} seconds")
    print(f"Heuristic objective: ${heuristic_cost:,.2f}")
    print(f"Heuristic solve time: {heuristic_time:.6f} seconds")
    if math.isfinite(gap_percent):
        print(f"Heuristic relative gap: {gap_percent:.2f}%")
    else:
        print("Heuristic relative gap: undefined because the exact objective is zero.")

    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    labels = ["Exact model", "Cheapest insertion"]
    colors = ["#1b9e77", "#d95f02"]

    cost_bars = axes[0].bar(labels, [exact_cost, heuristic_cost], color=colors)
    axes[0].set_ylabel("Expected cost ($)")
    axes[0].set_title("Objective comparison")
    axes[0].grid(axis="y", alpha=0.3)
    cost_offset = max(exact_cost, heuristic_cost, 0.0) * 0.01
    for bar, cost in zip(cost_bars, [exact_cost, heuristic_cost]):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + cost_offset,
            f"${cost:,.2f}",
            ha="center",
            va="bottom",
        )

    time_bars = axes[1].bar(
        labels,
        [exact_time, heuristic_time],
        color=colors,
    )
    axes[1].set_ylabel("Measured solve time (seconds)")
    axes[1].set_title("Time on this run")
    axes[1].grid(axis="y", alpha=0.3)
    for bar, elapsed in zip(time_bars, [exact_time, heuristic_time]):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{elapsed:.4f} s",
            ha="center",
            va="bottom",
        )

    figure.tight_layout()
    plt.show()


compare_with_exact_method()

## 6. Interpretation and limitations

The heuristic's result is a consequence of its sequence of local decisions. A low selection score does not measure the full future effect of an assignment: it does not anticipate which customers will remain or how scarce capacity may constrain their later placement. Consequently, the heuristic has no optimality guarantee and may stop with customers unassigned. A complete construction is feasible for the stated capacity constraints, but it is not necessarily the least-cost feasible solution.

Scenario weighting incorporates uncertainty through expected transportation cost, while the capacity checks protect feasibility in every listed scenario. These are distinct roles: probability weights affect the objective, whereas the model requires the capacity limit to hold scenario by scenario. Fixed facility costs are counted once, at the first assignment that opens each facility.

The capacity-relaxed reference is useful for showing the effect of ignoring capacity, but it is not a feasible benchmark. The exact comparison is more informative about solution quality only for its small instance and matching model. Measured run times depend on the computer, solver, and software environment; one run should not be treated as a general performance guarantee.

### Further study

The construction rule can be studied on additional instances and demand scenarios while retaining its greedy assignment logic. Natural modeling extensions include additional products, service requirements, or facility-specific operating limits. Each extension requires its assumptions and constraints to be stated explicitly; results from the present model should not be interpreted as accounting for features that are outside its scope.